In [2]:
!pip install -q rank_bm25 bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.0 MB/s eta 0:00:00:00:0100:01


In [3]:
# ==============================================================================
# CVR-LLM: Complex Visual Reasoning Large Language Models
# Implementation based on paper summary.
# Environments: Kaggle / Google Colab (Requires T4 GPU minimum)
# Install requirements before running:
# !pip install -q transformers torch torchvision rank_bm25 scikit-learn pandas bitsandbytes accelerate
# ==============================================================================

import os
import time
import torch
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from transformers import (
    AutoProcessor, Blip2ForConditionalGeneration, Blip2Model,
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
)
from rank_bm25 import BM25Okapi

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
class Config:
    # Model Paths
    # ASSUMPTION: To fit on Colab/Kaggle 16GB, we use flan-t5-xl instead of xxl, 
    # and a smaller LLM like Qwen-1.5B or a quantised Llama-3-8B.
    # The paper explicitly states BLIP2-flant5xxl and Llama3-8B/GPT.
    BLIP2_MODEL_NAME = "Salesforce/blip2-flan-t5-xl" # "Salesforce/blip2-flan-t5-xxl" (Paper)
    LLM_MODEL_NAME = "Qwen/Qwen1.5-1.8B-Chat" # Replace with "meta-llama/Meta-Llama-3-8B-Instruct" if granted access
    
    # Dataset Paths
    DATASET_TRAIN_PATH = "./data/train" # Dummy path
    DATASET_TEST_PATH = "./data/test"   # Dummy path
    OUTPUT_RESULT_CSV = "cvr_llm_results.csv"
    
    # CVR-ICL Hyperparameters (Explicitly stated in paper)
    K_EXAMPLES = 4
    ALPHA = 1.0
    
    # Hardware
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_4BIT = True # ASSUMPTION: Used to prevent OOM on free Colab/Kaggle

# ==============================================================================
# 2. LOAD DATASET (Dummy Data Generator for Runnability)
# ==============================================================================
def load_dataset():
    """
    Generates dummy binary classification VQA dataset.
    Since CVR datasets (Winoground, WinoGAVIL) are large, this ensures the code runs instantly.
    """
    print("Loading datasets...")
    # Generate dummy solid color images
    img1 = Image.new('RGB', (224, 224), color = 'red')
    img2 = Image.new('RGB', (224, 224), color = 'blue')
    
    train_pool = [
        {"id": "tr1", "image": img1, "task": "Is this image predominantly red? Answer Yes or No.", "label": "Yes", "label_binary": 1},
        {"id": "tr2", "image": img2, "task": "Is this image predominantly red? Answer Yes or No.", "label": "No", "label_binary": 0},
        {"id": "tr3", "image": img1, "task": "Does this image show a warm color? Answer Yes or No.", "label": "Yes", "label_binary": 1},
        {"id": "tr4", "image": img2, "task": "Does this image show a warm color? Answer Yes or No.", "label": "No", "label_binary": 0},
        {"id": "tr5", "image": img1, "task": "Is it blue? Answer Yes or No.", "label": "No", "label_binary": 0},
    ]
    
    test_set = [
        {"id": "te1", "image": img1, "task": "Is this image predominantly red? Answer Yes or No.", "label": "Yes", "label_binary": 1},
        {"id": "te2", "image": img2, "task": "Is this image predominantly red? Answer Yes or No.", "label": "No", "label_binary": 0},
    ]
    return train_pool, test_set

# ==============================================================================
# 3. MODELS & MODULES
# ==============================================================================
class ModelManager:
    def __init__(self, config):
        self.config = config
        self.device = config.DEVICE
        
        print(f"Loading Models onto {self.device}...")
        quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16) if config.USE_4BIT else None
        
        # 1. BLIP-2 Captioner
        self.blip_processor = AutoProcessor.from_pretrained(config.BLIP2_MODEL_NAME)
        self.blip_model = Blip2ForConditionalGeneration.from_pretrained(
            config.BLIP2_MODEL_NAME, 
            quantization_config=quantization_config,
            device_map="auto"
        )
        
        # 2. BLIP-2 Multi-modal Encoder (for embeddings)
        # ASSUMPTION: Paper states "BLIP2 multi-embedding". Blip2Model provides Q-Former embeddings.
        self.blip_encoder = Blip2Model.from_pretrained(
            config.BLIP2_MODEL_NAME,
            quantization_config=quantization_config,
            device_map="auto"
        )
        
        # 3. LLM (Llama3/GPT substitute)
        self.llm_tokenizer = AutoTokenizer.from_pretrained(config.LLM_MODEL_NAME)
        self.llm_model = AutoModelForCausalLM.from_pretrained(
            config.LLM_MODEL_NAME,
            quantization_config=quantization_config,
            device_map="auto"
        )

    def generate_llm_response(self, prompt, max_new_tokens=50):
        # ASSUMPTION: Paper does not specify exact generation hyperparameters (temperature, etc.)
        inputs = self.llm_tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.llm_model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.01)
        response = self.llm_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return response.strip()

    def generate_blip_caption(self, image, text_prompt):
        # ASSUMPTION: Paper does not specify BLIP generation kwargs (beam search vs greedy). Using greedy for speed.
        inputs = self.blip_processor(images=image, text=text_prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.blip_model.generate(**inputs, max_new_tokens=50)
        return self.blip_processor.decode(outputs[0], skip_special_tokens=True).strip()
        
    def get_blip_multimodal_embedding(self, image, text):
        # ASSUMPTION: Paper does not specify pooling strategy. We mean-pool Q-former outputs over sequence length.
        inputs = self.blip_processor(images=image, text=text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            qformer_out = self.blip_encoder.get_qformer_features(**inputs)
        return qformer_out.mean(dim=1).squeeze().cpu().numpy()

class CaID_Module:
    """Context-Aware Image Description (CaID) - Dual Loop Self-Refinement"""
    def __init__(self, manager: ModelManager):
        self.manager = manager
        
    def forward(self, image, task):
        # --- Loop 1: Initialization ---
        # ASSUMPTION: Paper does not specify exact prompt template for task extraction.
        l_t_prompt = f"Extract the core visual requirements from this task to guide an image captioning model. Task: '{task}'. Output only the requirements."
        extracted_prompt = self.manager.generate_llm_response(l_t_prompt)
        d_init = self.manager.generate_blip_caption(image, extracted_prompt)
        
        # --- Loop 2: Refinement ---
        # ASSUMPTION: Paper does not specify exact prompt template for preliminary prediction and feedback.
        feedback_prompt = (f"Task: '{task}'. Initial Image Description: '{d_init}'. "
                           f"Make a preliminary prediction and ask one specific question to look deeper into the image to refine the description.")
        feedback_q = self.manager.generate_llm_response(feedback_prompt)
        d_revised = self.manager.generate_blip_caption(image, feedback_q)
        
        return d_revised

class CVR_ICL_Module:
    """Complex Visual Reasoning In-Context Learning Selector"""
    def __init__(self, manager: ModelManager, config: Config):
        self.manager = manager
        self.config = config
        self.pool_data = []
        self.bm25_index = None
        self.multi_modal_embeddings = []
        
    def build_index(self, train_pool):
        """Preprocessing & Feature Selection for pool"""
        print("Building CVR-ICL Index from train pool...")
        self.pool_data = train_pool
        corpus_texts = []
        
        for item in train_pool:
            text_rep = f"{item['task']} {item['label']}"
            corpus_texts.append(text_rep.split()) # Simple whitespace tokenizer for BM25
            
            # Pre-compute BLIP-2 multi-modal embeddings
            emb = self.manager.get_blip_multimodal_embedding(item["image"], item["task"])
            self.multi_modal_embeddings.append(emb)
            
        self.bm25_index = BM25Okapi(corpus_texts)
        self.multi_modal_embeddings = np.array(self.multi_modal_embeddings)
        print("Index building completed.")

    def retrieve(self, test_image, test_task, test_d_revised):
        # 1. Text Score (BM25) - s_t
        tokenized_query = (test_task + " " + test_d_revised).split()
        s_t = self.bm25_index.get_scores(tokenized_query)
        
        # 2. Multi-modal Score (Cosine Sim) - s_m
        q_emb = self.manager.get_blip_multimodal_embedding(test_image, test_task)
        
        # Compute Cosine Similarity
        norms_pool = np.linalg.norm(self.multi_modal_embeddings, axis=1)
        norm_q = np.linalg.norm(q_emb)
        s_m = np.dot(self.multi_modal_embeddings, q_emb) / (norms_pool * norm_q + 1e-10)
        
        # 3. Combine Score
        # ASSUMPTION: Paper does not specify if BM25 and Cosine scores are normalized before addition.
        # BM25 scores can be arbitrarily large, Cosine is [-1, 1]. We perform Min-Max scaling to [0,1] for both to make alpha=1 meaningful.
        s_t_scaled = (s_t - np.min(s_t)) / (np.max(s_t) - np.min(s_t) + 1e-10)
        s_m_scaled = (s_m - np.min(s_m)) / (np.max(s_m) - np.min(s_m) + 1e-10)
        
        s = self.config.ALPHA * s_m_scaled + s_t_scaled
        
        # Get top K
        top_k_indices = np.argsort(s)[-self.config.K_EXAMPLES:][::-1]
        
        examples = [self.pool_data[i] for i in top_k_indices]
        return examples

# ==============================================================================
# 4. TRAINING & EVALUATION PIPELINE
# ==============================================================================
def run_pipeline():
    config = Config()
    train_pool, test_set = load_dataset()
    
    manager = ModelManager(config)
    caid = CaID_Module(manager)
    cvr_icl = CVR_ICL_Module(manager, config)
    
    # --- "Training" Phase ---
    # Inference-only method. "Training time" is just indexing time.
    start_train = time.time()
    cvr_icl.build_index(train_pool)
    train_time = time.time() - start_train
    
    # Calculate approx params (For print requirements)
    total_params = sum(p.numel() for p in manager.blip_model.parameters()) + \
                   sum(p.numel() for p in manager.llm_model.parameters())
    
    # --- Evaluation Phase ---
    print("Starting Inference on Test Set...")
    y_true = []
    y_pred_binary = []
    y_pred_probs = [] # Mock probability for AUC
    
    for i, item in enumerate(test_set):
        print(f"Processing {i+1}/{len(test_set)}: {item['id']}")
        
        # 1. CaID Module
        d_revised = caid.forward(item["image"], item["task"])
        
        # 2. CVR-ICL Retrieval
        top_k_examples = cvr_icl.retrieve(item["image"], item["task"], d_revised)
        
        # 3. Final LLM Prediction
        # Build prompt from ICL examples
        icl_context = "\n".join([f"Q: {ex['task']} A: {ex['label']}" for ex in top_k_examples])
        
        # ASSUMPTION: Paper doesn't give exact final prompt. Creating a standard few-shot prompt.
        final_prompt = (
            f"You are a visual reasoning assistant. Answer with exactly 'Yes' or 'No'.\n"
            f"Here are some examples:\n{icl_context}\n\n"
            f"Image Description: {d_revised}\n"
            f"Q: {item['task']}\nA:"
        )
        
        final_ans = manager.generate_llm_response(final_prompt, max_new_tokens=5)
        
        # Parse binary prediction
        pred_label = 1 if "yes" in final_ans.lower() else 0
        
        y_true.append(item["label_binary"])
        y_pred_binary.append(pred_label)
        # Mock probability generation based on final answer confidence
        y_pred_probs.append(0.9 if pred_label == 1 else 0.1) 
        
    # --- Calculate Metrics ---
    # Ensure there are both classes in true_labels to avoid AUC error on dummy data
    if len(set(y_true)) > 1:
        acc = accuracy_score(y_true, y_pred_binary)
        prec = precision_score(y_true, y_pred_binary, zero_division=0)
        rec = recall_score(y_true, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        roc_auc = roc_auc_score(y_true, y_pred_probs)
        pr_auc = average_precision_score(y_true, y_pred_probs)
        
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    else:
        # Fallback if dummy data test set only has 1 class
        acc, prec, rec, f1, roc_auc, pr_auc, fpr, fnr = (0,)*8
        acc = accuracy_score(y_true, y_pred_binary)

    metrics = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "FPR": fpr,
        "FNR": fnr,
        "Train_Time_sec": train_time,
        "Total_Params": total_params
    }
    
    print("\n--- FINAL METRICS ---")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    # ==============================================================================
    # 5. SAVE RESULTS
    # ==============================================================================
    df_results = pd.DataFrame([metrics])
    df_results.to_csv(config.OUTPUT_RESULT_CSV, index=False)
    print(f"\nResults saved to {config.OUTPUT_RESULT_CSV}")

if __name__ == "__main__":
    run_pipeline()

Loading datasets...
Loading Models onto cuda...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

Building CVR-ICL Index from train pool...


/tmp/ipykernel_58/152617079.py:130: UserWarning: The following named arguments are not valid for `Blip2Model.get_qformer_features` and were ignored: 'input_ids', 'attention_mask'
  qformer_out = self.blip_encoder.get_qformer_features(**inputs)


Index building completed.
Starting Inference on Test Set...
Processing 1/2: te1
Processing 2/2: te2

--- FINAL METRICS ---
Accuracy: 0.5
Precision: 0.5
Recall: 1.0
F1: 0.6666666666666666
ROC-AUC: 0.5
PR-AUC: 0.5
FPR: 1.0
FNR: 0.0
Train_Time_sec: 2.3665125370025635
Total_Params: 3320397312

Results saved to cvr_llm_results.csv
